In [0]:
%sql
USE CATALOG test_analytics;
USE SCHEMA raw;

In [0]:
from typing import Optional, Dict, Any, List
import time, os, json, math
import requests
from pyspark.sql import SparkSession, DataFrame
import pandas as pd
from datetime import datetime
import numpy as np
import pathlib

In [0]:
class DeepseekSegmentAnalyst:
    """
    Aggregates churn segments and asks DeepSeek (chat/completions) for exec insights.
    Uses raw HTTP to ensure we always hit DeepSeek, not OpenAI.
    """

    def __init__(
        self,
        spark: SparkSession,
        secret_scope: str = "deepseek-secrets",
        secret_key_name: str = "API_KEY",
        *,
        model: str = "deepseek-chat",          # or "deepseek-reasoner"
        base_url: str = "https://api.deepseek.com/v1",
        temperature: float = 0.2,
        max_tokens: int = 600,
        timeout: int = 60,
        max_retries: int = 5,
        initial_backoff_sec: float = 1.5,
        api_key_override: Optional[str] = None,
        log_prefix: str = "[DeepseekSegmentAnalyst]"
    ):
        self.spark = spark
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.timeout = timeout
        self.max_retries = max_retries
        self.initial_backoff_sec = initial_backoff_sec
        self.log_prefix = log_prefix

        # Resolve API key from Databricks Secrets unless overridden.
        api_key = api_key_override
        if api_key is None:
            try:
                api_key = dbutils.secrets.get(secret_scope, secret_key_name)  # noqa: F821 (available on DBX)
            except Exception as e:
                raise RuntimeError(
                    f"{self.log_prefix} Could not read secret '{secret_key_name}' "
                    f"from scope '{secret_scope}'. Set api_key_override or fix secrets. Err: {e}"
                )
        self.api_key = api_key

        self._session = requests.Session()
        self._session.headers.update({
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        })

    # -----------------------------
    # Utils
    # -----------------------------
    def verify_connection(self) -> None:
        """Quick check: list models to validate URL + key."""
        url = f"{self.base_url}/models"
        try:
            r = self._session.get(url, timeout=self.timeout)
            if r.status_code == 200:
                print(f"{self.log_prefix} DeepSeek connection OK. Models: {[m.get('id') for m in r.json().get('data', [])][:5]}")
            else:
                print(f"{self.log_prefix} Verify failed ({r.status_code}): {r.text}")
        except Exception as e:
            print(f"{self.log_prefix} Verify error: {e}")

    # -----------------------------
    # 1) Build the segments DataFrame
    # -----------------------------
    def build_segments(
        self,
        source_table: str = "churn_predictions_raw",
        pipeline_run_id: Optional[str] = None,
    ) -> DataFrame:
        where_clause = f"WHERE pipeline_run_id = '{pipeline_run_id}'" if pipeline_run_id else ""
        query = f"""
            SELECT 
              CASE 
                WHEN prediction = 1 AND recency > 180 THEN 'Dormant_HighRisk'
                WHEN prediction = 1 AND monetary > 500 THEN 'HighValue_AtRisk' 
                WHEN prediction = 1 AND frequency < 3 THEN 'LowEngagement_Risk'
                WHEN prediction = 1 AND tenure_days < 90 THEN 'NewCustomer_Risk'
                WHEN prediction = 1 THEN 'General_AtRisk'
                WHEN prediction = 0 AND monetary > 1000 THEN 'HighValue_Safe'
                ELSE 'LowRisk_Stable'
              END as segment_name,

              COUNT(*) as customer_count,
              ROUND(AVG(recency), 1) as avg_recency_days,
              ROUND(AVG(frequency), 1) as avg_frequency,
              ROUND(AVG(monetary), 2) as avg_monetary_value,
              ROUND(AVG(tenure_days), 1) as avg_tenure_days,
              ROUND(AVG(CAST(get_json_object(probability_vector, '$[1]') AS DOUBLE)), 3)
                as avg_churn_probability,
              SUM(monetary) as total_customer_value,

              COUNT(CASE WHEN recency > 120 THEN 1 END) as count_inactive_120plus,
              COUNT(CASE WHEN monetary > 500 THEN 1 END) as count_high_value,
              COUNT(CASE WHEN frequency < 2 THEN 1 END) as count_low_engagement

            FROM {source_table}
            {where_clause}
            GROUP BY 1
        """
        return self.spark.sql(query)

    # -----------------------------
    # 2) Prompt construction
    # -----------------------------
    @staticmethod
    def _make_prompt(row: Dict[str, Any]) -> str:
        seg = row.get("segment_name", "Unknown")
        cc  = int(row.get("customer_count", 0))
        ar  = float(row.get("avg_recency_days", 0.0))
        af  = float(row.get("avg_frequency", 0.0))
        am  = float(row.get("avg_monetary_value", 0.0))
        at  = float(row.get("avg_tenure_days", 0.0))
        ap  = float(row.get("avg_churn_probability", 0.0))
        tv  = float(row.get("total_customer_value", 0.0))
        c120 = int(row.get("count_inactive_120plus", 0))
        chv  = int(row.get("count_high_value", 0))
        cle  = int(row.get("count_low_engagement", 0))

        return f"""
You are a Strategic Customer Retention AI Agent. Provide executive-ready insights in ~5-6 sentences, crisp and actionable.

SEGMENT ANALYSIS
- Segment: {seg}
- Size: {cc} customers
- Total Value at Risk: ${tv:,.2f}
- Average Churn Probability: {ap*100:.1f}%

KEY CHARACTERISTICS
- Avg Days Since Last Activity: {ar:.1f}
- Avg Purchase Frequency: {af:.1f}
- Avg Customer Value: ${am:.2f}
- Avg Tenure: {at:.1f} days
- Highly Inactive (>120 days): {c120}
- High-Value (>$500): {chv}
- Low Engagement (<2 purchases): {cle}

Deliver:
1) Risk Assessment  2) Primary Drivers  3) 2–3 Targeted Actions  4) Priority & Timeline  5) Expected Impact.
Keep it concise, data-driven, and business-focused.
""".strip()

    @staticmethod
    def _system_message() -> str:
        return ("You are a senior customer retention strategist. "
                "Be decisive, quantify impact where possible, and avoid generic advice.")

    # -----------------------------
    # 3) DeepSeek chat call with retries
    # -----------------------------
    def _chat_completions(self, messages: List[Dict[str, str]]) -> str:
        url = f"{self.base_url}/chat/completions"
        payload = {
            "model": self.model,
            "messages": messages,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }

        backoff = self.initial_backoff_sec
        last_err = None
        for attempt in range(1, self.max_retries + 1):
            try:
                r = self._session.post(url, data=json.dumps(payload), timeout=self.timeout)
                if r.status_code == 200:
                    j = r.json()
                    return (j["choices"][0]["message"]["content"] or "").strip()

                # Retryable?
                if r.status_code in (408, 409, 429, 500, 502, 503, 504):
                    print(f"{self.log_prefix} Retryable error {r.status_code} (attempt {attempt}): {r.text}")
                else:
                    # Non-retryable (401/403/etc.) -> print & break
                    print(f"{self.log_prefix} Non-retryable error {r.status_code}: {r.text}")
                    return f"DeepSeek error ({r.status_code}): {r.text}"

            except Exception as e:
                last_err = e
                print(f"{self.log_prefix} Exception (attempt {attempt}/{self.max_retries}): {e}")

            if attempt < self.max_retries:
                time.sleep(backoff)
                backoff *= 2

        return f"Analysis temporarily unavailable. Last error: {last_err}"

    def _call_deepseek(self, prompt: str, system_message: Optional[str] = None) -> str:
        sm = system_message or self._system_message()
        messages = [
            {"role": "system", "content": sm},
            {"role": "user", "content": prompt},
        ]
        return self._chat_completions(messages)

    # -----------------------------
    # 4) Analyze one segment row
    # -----------------------------
    def analyze_row(self, row: Dict[str, Any]) -> Dict[str, Any]:
        prompt = self._make_prompt(row)
        insight = self._call_deepseek(prompt)
        out = dict(row)
        out["executive_insight"] = insight
        return out

    # -----------------------------
    # 5) Analyze all segments
    # -----------------------------
    def analyze_segments(self, segments_df: DataFrame, throttle_sec: float = 0.0) -> pd.DataFrame:
        rows = [r.asDict(recursive=True) for r in segments_df.collect()]
        results: List[Dict[str, Any]] = []
        for idx, r in enumerate(rows, start=1):
            print(f"{self.log_prefix} Analyzing segment {idx}/{len(rows)}: {r.get('segment_name')}")
            results.append(self.analyze_row(r))
            if throttle_sec > 0 and idx < len(rows):
                time.sleep(throttle_sec)
        return pd.DataFrame(results)

    # -----------------------------
    # 6) Optional: write to Delta
    # -----------------------------
    def write_results_delta(self, results_pdf: pd.DataFrame, target_table: str, mode: str = "append"):
        self.spark.createDataFrame(results_pdf).write.format("delta").mode(mode).saveAsTable(target_table)
        print(f"{self.log_prefix} Wrote {len(results_pdf)} rows to {target_table} (mode={mode}).")

    # -----------------------------
    # 7) Optional: generate html report
    # -----------------------------

    def render_html_report(
    self,
    results_pdf: pd.DataFrame,
    *,
    title: str = "Customer Churn — Executive Segment Insights",
    company_name: str = "Your Company",
    output_path: str = "./churn_segment_insights.html",
    dark_mode: bool = True
) -> (str, str):
        """
        Renders an HTML report for execs and writes it to DBFS.
        Returns (dbfs_path, public_url).
        - Anything under /dbfs/FileStore/... is publicly reachable at /files/...
        """

        if results_pdf is None or results_pdf.empty:
            raise ValueError("results_pdf is empty; run analyze_segments() first.")

        # KPIs
        total_segments = len(results_pdf)
        total_customers = int(results_pdf["customer_count"].sum())
        total_value = float(results_pdf["total_customer_value"].sum())
        weighted_churn = float(np.average(
            results_pdf["avg_churn_probability"],
            weights=results_pdf["customer_count"].clip(lower=1)
        ))

        # Sort segments by risk (highest churn first)
        df = results_pdf.copy()
        df = df.sort_values(["avg_churn_probability", "total_customer_value"], ascending=[False, False])

        # Color tag by segment_name (simple mapping)
        badge_map = {
            "Dormant_HighRisk": "#ef4444",       # red
            "HighValue_AtRisk": "#f97316",       # orange
            "LowEngagement_Risk": "#fb923c",     # amber
            "NewCustomer_Risk": "#06b6d4",       # cyan
            "General_AtRisk": "#f59e0b",         # yellow
            "HighValue_Safe": "#22c55e",         # green
            "LowRisk_Stable": "#84cc16",         # lime
        }

        # Minimal CSS (self-contained)
        base_bg = "#0b1220" if dark_mode else "#ffffff"
        base_fg = "#e5e7eb" if dark_mode else "#0f172a"
        card_bg = "#111827" if dark_mode else "#f8fafc"
        border_c = "#1f2937" if dark_mode else "#e5e7eb"
        accent = "#60a5fa"

        def fmt_money(x): return f"${x:,.2f}"
        def fmt_pct(x):   return f"{x*100:.1f}%"

        # Build rows
        row_html = []
        for _, r in df.iterrows():
            seg = r["segment_name"]
            color = badge_map.get(seg, accent)
            row_html.append(f"""
        <tr>
          <td>
            <span style="background:{color};color:white;padding:4px 10px;border-radius:9999px;font-weight:600;">
              {seg}
            </span>
          </td>
          <td style="text-align:right;">{int(r["customer_count"]):,}</td>
          <td style="text-align:right;">{r["avg_recency_days"]:.1f}</td>
          <td style="text-align:right;">{r["avg_frequency"]:.1f}</td>
          <td style="text-align:right;">{fmt_money(float(r["avg_monetary_value"]))}</td>
          <td style="text-align:right;">{r["avg_tenure_days"]:.1f}</td>
          <td style="text-align:right;font-weight:600;">{fmt_pct(float(r["avg_churn_probability"]))}</td>
          <td style="text-align:right;">{fmt_money(float(r["total_customer_value"]))}</td>
        </tr>
        <tr>
          <td colspan="8" style="padding:12px 16px;background:{card_bg};border:1px solid {border_c};border-radius:10px;">
            <div style="font-size:14px;line-height:1.5;color:{base_fg};white-space:pre-wrap;">{r["executive_insight"]}</div>
          </td>
        </tr>
        """)

        generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{title}</title>
<style>
  body {{
    background:{base_bg};
    color:{base_fg};
    font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial, "Apple Color Emoji", "Segoe UI Emoji";
    margin:0; padding:24px;
  }}
  .container {{ max-width:1100px; margin:0 auto; }}
  .card {{
    background:{card_bg};
    border:1px solid {border_c};
    border-radius:16px;
    padding:20px;
    margin-top:16px;
  }}
  h1 {{ margin:0; font-size:28px; }}
  h2 {{ margin:0 0 10px 0; font-size:20px; }}
  .kpis {{
    display:grid; grid-template-columns: repeat(4, 1fr);
    gap:12px; margin-top:16px;
  }}
  .kpi .label {{ font-size:12px; opacity:.8; }}
  .kpi .val {{ font-size:20px; font-weight:700; }}
  table {{
    width:100%; border-collapse:separate; border-spacing:0 10px; margin-top:10px;
  }}
  th, td {{ padding:10px 12px; }}
  th {{
    text-align:left; font-size:12px; letter-spacing:.02em; text-transform:uppercase; opacity:.75;
    border-bottom:1px solid {border_c};
  }}
  tr td {{
    background:transparent;
  }}
  .footer {{ margin-top:24px; font-size:12px; opacity:.75; }}
  .brand {{ color:{accent}; font-weight:700; }}
</style>
</head>
<body>
  <div class="container">
    <div class="card">
      <h1>{title}</h1>
      <div style="margin-top:6px;">Prepared for <span class="brand">{company_name}</span></div>
      <div class="kpis">
        <div class="kpi"><div class="label">Segments</div><div class="val">{total_segments}</div></div>
        <div class="kpi"><div class="label">Customers Covered</div><div class="val">{total_customers:,}</div></div>
        <div class="kpi"><div class="label">Weighted Avg Churn</div><div class="val">{fmt_pct(weighted_churn)}</div></div>
        <div class="kpi"><div class="label">Total Value</div><div class="val">{fmt_money(total_value)}</div></div>
      </div>
    </div>

    <div class="card">
      <h2>Segment Insights</h2>
      <table>
        <thead>
          <tr>
            <th>Segment</th>
            <th style="text-align:right;">Customers</th>
            <th style="text-align:right;">Avg Recency (d)</th>
            <th style="text-align:right;">Avg Freq</th>
            <th style="text-align:right;">Avg Value</th>
            <th style="text-align:right;">Avg Tenure (d)</th>
            <th style="text-align:right;">Avg Churn</th>
            <th style="text-align:right;">Total Value</th>
          </tr>
        </thead>
        <tbody>
          {''.join(row_html)}
        </tbody>
      </table>
    </div>

    <div class="footer">Generated {generated_at}</div>
  </div>
</body>
</html>"""

        # Ensure parent dir exists and write
        pathlib.Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(html)

        # Public URL mapping for FileStore
        public_url = output_path.replace("/dbfs/FileStore", "/files")
        print(f"{self.log_prefix} HTML report written to: {output_path}")
        print(f"{self.log_prefix} Public URL (in workspace): {public_url}")
        return output_path, public_url

